# Step 5: Adam

Bu notebook'ta Step 4'te oluşturduğum Transformer modelini Adam optimizer ile eğiteceğim.
Model ve veri kümesi aynı kalacak; bu bölümde eğitim sırasında parametrelerin nasıl güncellendiği değişecek.

In [1]:
import random
import math

random.seed(42)

with open("../data/raw_names.txt", encoding="utf-8") as f:
    docs = [
        line.strip()
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .replace(" ", "")
        for line in f
        if line.strip()
    ]

random.shuffle(docs)

uchars = sorted(set("".join(docs)))
stoi = {ch: i for i, ch in enumerate(uchars)}

BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Karakter sayısı:", len(uchars))
print("Sözlük boyutu:", vocab_size)
print("BOS tokenı:", BOS)

İsim sayısı: 1100
Karakter sayısı: 29
Sözlük boyutu: 30
BOS tokenı: 29


In [2]:
class Value:
    __slots__ = ("data", "grad", "_children", "_local_grads")

    def __init__(self, data, children=(), local_grads=()):
        self.data = float(data)
        self.grad = 0.0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data + other.data,
            (self, other),
            (1.0, 1.0),
        )

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data),
        )

    def __pow__(self, power):
        return Value(
            self.data ** power,
            (self,),
            (power * self.data ** (power - 1),),
        )

    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1.0 / self.data,),
        )

    def exp(self):
        result = math.exp(self.data)
        return Value(
            result,
            (self,),
            (result,),
        )

    def relu(self):
        return Value(
            max(0.0, self.data),
            (self,),
            (float(self.data > 0),),
        )

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)

                for child in node._children:
                    build_topo(child)

                topo.append(node)

        build_topo(self)
        self.grad = 1.0

        for node in reversed(topo):
            for child, local_grad in zip(
                node._children,
                node._local_grads
            ):
                child.grad += local_grad * node.grad

In [3]:
def linear(x, w):
    return [
        sum(weight * value for weight, value in zip(row, x))
        for row in w
    ]


def softmax(logits):
    max_val = max(value.data for value in logits)
    exps = [(value - max_val).exp() for value in logits]
    total = sum(exps)
    return [value / total for value in exps]


def rmsnorm(x):
    mean_square = sum(value * value for value in x) / len(x)
    scale = (mean_square + 1e-5) ** -0.5
    return [value * scale for value in x]

In [4]:
n_embd = 16
block_size = 16
n_head = 4
n_layer = 1

assert n_embd % n_head == 0
head_dim = n_embd // n_head

matrix = lambda nout, nin, std=0.08: [
    [Value(random.gauss(0, std)) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    "wte": matrix(vocab_size, n_embd),
    "wpe": matrix(block_size, n_embd),
    "lm_head": matrix(vocab_size, n_embd),
}

for i in range(n_layer):
    state_dict[f"layer{i}.attn_wq"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wk"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wv"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wo"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc1"] = matrix(4 * n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc2"] = matrix(n_embd, 4 * n_embd)

def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict["wte"][token_id]
    pos_emb = state_dict["wpe"][pos_id]
    x = [token + position for token, position in zip(tok_emb, pos_emb)]

    for li in range(n_layer):
        # Multi-head attention
        x_residual = x
        x = rmsnorm(x)

        q = linear(x, state_dict[f"layer{li}.attn_wq"])
        k = linear(x, state_dict[f"layer{li}.attn_wk"])
        v = linear(x, state_dict[f"layer{li}.attn_wv"])

        keys[li].append(k)
        values[li].append(v)

        x_attn = []

        for h in range(n_head):
            hs = h * head_dim

            q_h = q[hs:hs + head_dim]
            k_h = [key[hs:hs + head_dim] for key in keys[li]]
            v_h = [value[hs:hs + head_dim] for value in values[li]]

            attn_logits = [
                sum(q_h[j] * k_h[t][j] for j in range(head_dim))
                / head_dim ** 0.5
                for t in range(len(k_h))
            ]

            attn_weights = softmax(attn_logits)

            head_out = [
                sum(
                    attn_weights[t] * v_h[t][j]
                    for t in range(len(v_h))
                )
                for j in range(head_dim)
            ]

            x_attn.extend(head_out)

        x = linear(x_attn, state_dict[f"layer{li}.attn_wo"])
        x = [a + b for a, b in zip(x, x_residual)]

        # MLP
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f"layer{li}.mlp_fc1"])
        x = [value.relu() for value in x]
        x = linear(x, state_dict[f"layer{li}.mlp_fc2"])
        x = [a + b for a, b in zip(x, x_residual)]

    return linear(x, state_dict["lm_head"])
params = [
    parameter
    for matrix_values in state_dict.values()
    for row in matrix_values
    for parameter in row
]

print("Attention başlığı sayısı:", n_head)
print("Başlık başına boyut:", head_dim)
print("Katman sayısı:", n_layer)
print("Parametre sayısı:", len(params))

Attention başlığı sayısı: 4
Başlık başına boyut: 4
Katman sayısı: 1
Parametre sayısı: 4288


## 5.2 Momentum ve Uyarlanabilir Öğrenme Oranı

Adam, her parametre için iki değer tutar: `m`, gradyanların yönünü yumuşatır; `v`, büyüklüklerini izleyerek adım boyutunu ayarlar.

\[
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
\qquad
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
\]

İkisi de sıfırdan başladığı için Adam ilk adımlarda bias düzeltmesi uygular. Aşağıdaki örnek bu hesaplamaları gösterir;
model ağırlıklarını güncellemez.

In [5]:
beta1 = 0.85
beta2 = 0.99

gradients = [2.0, -1.0, 3.0]
m = 0.0
v = 0.0

for step, g in enumerate(gradients, start=1):
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * (g ** 2)

    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)

    print(
        f"Adım {step}: gradyan={g:.1f}, "
        f"m={m:.4f}, v={v:.4f}, "
        f"düzeltilmiş m={m_hat:.4f}, "
        f"düzeltilmiş v={v_hat:.4f}"
    )

Adım 1: gradyan=2.0, m=0.3000, v=0.0400, düzeltilmiş m=2.0000, düzeltilmiş v=4.0000
Adım 2: gradyan=-1.0, m=0.1050, v=0.0496, düzeltilmiş m=0.3784, düzeltilmiş v=2.4925
Adım 3: gradyan=3.0, m=0.5393, v=0.1391, düzeltilmiş m=1.3975, düzeltilmiş v=4.6835


## 5.3 Adam Kodunu Hazırlama

Adam, her parametre için iki değer saklar: `m` gradyanların yönünü, `v` ise büyüklüğünü izler. Bu değerler eğitim başlamadan önce sıfırlarla başlatılır.

Her eğitim adımında Adam, `m` ve `v` değerlerini günceller, başlangıç yanlılığını düzeltir ve parametreyi yeni gradyana göre ayarlar.

In [6]:
learning_rate = 0.01
beta1 = 0.85
beta2 = 0.99
eps_adam = 1e-8

m = [0.0] * len(params)
v = [0.0] * len(params)

print("Parametre sayısı:", len(params))
print("m eleman sayısı:", len(m))
print("v eleman sayısı:", len(v))

Parametre sayısı: 4288
m eleman sayısı: 4288
v eleman sayısı: 4288


## 5.4 Eğitim ve Sonuçlar

Her adımda bir isim tokenlara çevrilir ve modelin loss değeri hesaplanır.
`loss.backward()` gradyanları bulur. Adam, bu gradyanları `m` ve `v`
değerleriyle kullanarak model parametrelerini günceller.

Eğitimden sonra modelden yeni isimler üretirim.

In [7]:
num_steps = 1000

for step in range(num_steps):
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Bu isme ait cache'leri başlat
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]

    # Forward pass: loss hesapla
    losses = []

    for pos_id in range(n):
        token_id = tokens[pos_id]
        target_id = tokens[pos_id + 1]

        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)

    loss = (1 / n) * sum(losses)

    # Backward pass: gradyanları hesapla
    loss.backward()

    # Adam ile parametreleri güncelle
    lr_t = learning_rate * (1 - step / num_steps)

    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2

        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))

        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0.0

    if step % 100 == 0 or step == num_steps - 1:
        print(
            f"Adım: {step + 1:4d} | "
            f"Loss: {loss.data:.4f} | "
            f"Learning rate: {lr_t:.5f}"
        )

Adım:    1 | Loss: 3.3520 | Learning rate: 0.01000
Adım:  101 | Loss: 2.1133 | Learning rate: 0.00900
Adım:  201 | Loss: 2.6417 | Learning rate: 0.00800
Adım:  301 | Loss: 2.7902 | Learning rate: 0.00700
Adım:  401 | Loss: 2.5802 | Learning rate: 0.00600
Adım:  501 | Loss: 2.7136 | Learning rate: 0.00500
Adım:  601 | Loss: 3.2411 | Learning rate: 0.00400
Adım:  701 | Loss: 2.2840 | Learning rate: 0.00300
Adım:  801 | Loss: 2.1072 | Learning rate: 0.00200
Adım:  901 | Loss: 2.3621 | Learning rate: 0.00100
Adım: 1000 | Loss: 2.5028 | Learning rate: 0.00001


In [8]:
temperature = 0.5

for sample_idx in range(20):
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]

    token_id = BOS
    sample = []

    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([logit / temperature for logit in logits])

        token_id = random.choices(
            range(vocab_size),
            weights=[prob.data for prob in probs]
        )[0]

        if token_id == BOS:
            break

        sample.append(uchars[token_id])

    print(f"İsim {sample_idx + 1:2d}: {''.join(sample)}")

İsim  1: celiy
İsim  2: hanen
İsim  3: sera
İsim  4: sura
İsim  5: hana
İsim  6: oran
İsim  7: melin
İsim  8: elem
İsim  9: alabe
İsim 10: bela
İsim 11: elas
İsim 12: hamil
İsim 13: hari
İsim 14: reliy
İsim 15: murel
İsim 16: tene
İsim 17: halir
İsim 18: hari
İsim 19: hanay
İsim 20: sela
